In [42]:
import pandas as pd
import numpy as np
from imblearn.over_sampling import SMOTE
from pycaret.classification import *
import warnings
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
import numpy as np



In [43]:
warnings.filterwarnings("ignore")

# 데이터 로드
train_df = pd.read_csv("open/train.csv").drop(columns=['UID'])
test_df = pd.read_csv("open/test.csv").drop(columns=['UID'])

# 타겟 변수 확인
print(train_df['채무 불이행 여부'].value_counts(normalize=True))

채무 불이행 여부
0    0.6588
1    0.3412
Name: proportion, dtype: float64


In [9]:

# X, y 분리
X = train_df.drop(columns=['채무 불이행 여부'])
y = train_df['채무 불이행 여부']

In [44]:
# 로그 변환
log_columns = ["현재 미상환 신용액", "월 상환 부채액", "현재 대출 잔액"]
for col in log_columns:
    X[col] = np.log1p(X[col])
    test_df[col] = np.log1p(test_df[col])


In [45]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# # X, y 분리
# X = train_df.drop(columns=['채무 불이행 여부'])
# y = train_df['채무 불이행 여부']

# 범주형 변수 처리 (One-Hot Encoding)
categorical_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=categorical_cols)

# 훈련/검증 데이터 분할
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# SMOTE 적용 (데이터 불균형 해결)
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 확인
print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_smote))


Before SMOTE: [5270 2730]
After SMOTE: [5270 5270]


In [46]:
rf_model = RandomForestClassifier(
    bootstrap=True, ccp_alpha=0.0, class_weight=None,
    criterion='gini', max_depth=None, max_features='sqrt',
    max_leaf_nodes=None, max_samples=None,
    min_impurity_decrease=0.0, min_samples_leaf=1,
    min_samples_split=2, min_weight_fraction_leaf=0.0,
    monotonic_cst=None, n_estimators=100, n_jobs=-1,
    oob_score=False, random_state=42, verbose=0,
    warm_start=False
)
rf_model.fit(X_train_smote, y_train_smote)


RandomForestClassifier(n_jobs=-1, random_state=42)

In [47]:
# 범주형 변수 처리 (One-Hot Encoding)
categorical_cols = test_df.select_dtypes(include=['object']).columns
test_df = pd.get_dummies(test_df, columns=categorical_cols)

In [48]:
# 채무 불이행 '확률'을 예측합니다.
preds = rf_model.predict_proba(test_df)[:,1]

submit = pd.read_csv('open/sample_submission.csv')

# 결과 저장
submit['채무 불이행 확률'] = preds
submit.to_csv('rf6_rf.csv', encoding='UTF-8-sig', index=False)

In [30]:
param_grid = {
    'n_estimators': [50, 100, 150, 200, 250, 300],  # 트리 개수
    'max_depth': [None, 5, 10, 20, 30, 50, 100],  # 최대 깊이
    'min_samples_split': [2, 5, 10, 20, 50],  # 최소 분할 샘플 수
    'min_samples_leaf': [1, 2, 4, 10, 20],  # 최소 리프 노드 샘플 수
    # 'max_features': ['sqrt', 'log2', None],  # 사용 가능한 최대 피처 수
    'max_features': ['sqrt'],
    'bootstrap': [True, False],  # 부트스트랩 샘플링 여부
    'criterion': ['gini', 'entropy'],  # 불순도 계산 방식
    'class_weight': [None, 'balanced', 'balanced_subsample']  # 클래스 가중치 조정
    
}

In [37]:

grid_search = GridSearchCV(RandomForestClassifier(random_state=42),
                           param_grid, cv=5, n_jobs=-1)

# 4. 그리드 탐색 수행
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)


Best Parameters: {'bootstrap': True, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}
Best CV Score: 0.725


In [38]:
best_model = grid_search.best_estimator_

In [32]:
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42),
                                   param_distributions=param_grid, n_iter=500,
                                   cv=5, n_jobs=-1, scoring='roc_auc', verbose=1, random_state=42)

random_search.fit(X_train, y_train)

# 6. 최적의 하이퍼파라미터 및 성능 출력
grid_search_results = {

    "Best Params (RandomSearch)": random_search.best_params_,
    "Best CV Score (RandomSearch)": random_search.best_score_,

}


Fitting 5 folds for each of 500 candidates, totalling 2500 fits


In [33]:
grid_search_results

{'Best Params (RandomSearch)': {'n_estimators': 300,
  'min_samples_split': 20,
  'min_samples_leaf': 20,
  'max_features': 'sqrt',
  'max_depth': 30,
  'criterion': 'entropy',
  'class_weight': 'balanced_subsample',
  'bootstrap': True},
 'Best CV Score (RandomSearch)': 0.7440867165724852}

In [34]:
best_model = random_search.best_estimator_

In [35]:
best_model

RandomForestClassifier(class_weight='balanced_subsample', criterion='entropy',
                       max_depth=30, min_samples_leaf=20, min_samples_split=20,
                       n_estimators=300, random_state=42)

In [21]:
# 범주형 변수 처리 (One-Hot Encoding)
categorical_cols = test_df.select_dtypes(include=['object']).columns
test_df = pd.get_dummies(test_df, columns=categorical_cols)

In [39]:
# 채무 불이행 '확률'을 예측합니다.
preds = best_model.predict_proba(test_df)[:,1]

submit = pd.read_csv('open/sample_submission.csv')

# 결과 저장
submit['채무 불이행 확률'] = preds
submit.to_csv('rf5_rs_gs.csv', encoding='UTF-8-sig', index=False)